In [1]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 831, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 831 (delta 13), reused 20 (delta 6), pack-reused 791 (from 1)
Receiving objects: 100% (831/831), 5.28 MiB | 18.90 MiB/s, done.
Resolving deltas: 100% (273/273), done.
Mounted at /content/drive


In [ ]:
!pip install torch_geometric -q

In [3]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [4]:
from utils.datasets     import NF_IDS_Dataset
from utils.models       import EdgeGRU_Baseline_NoX
from utils.metrics      import calculate_metrics_gnn
from utils.training     import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment   import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots


# Auxiliary

In [5]:
ROOT_PATH = "./dataset_processed"

In [6]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [7]:
ROOT_DIR = "./results_earlystopping"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")

# Main

## Seeds

In [8]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

#NODE_DIM = 16
EDGE_DIM = 32   # 7 dst_port + 5 protocol + 20 numeric
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [9]:
model_config = {
    "model_name": None,
    "type": "EdgeGRU_Baseline_NoX",
    "model_params": {
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [10]:
EXPERIMENT_NAME="EdgeGRU_NoX_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [12]:
run_multiple_seeds(
    model_class=EdgeGRU_Baseline_NoX,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: EdgeGRU_NoX_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=EdgeGRU_NoX_BiasOn_seed42_20260419_163345

EdgeGRU_NoX_BiasOn_seed42
   Ep 1 | Loss: 0.2018 | Val Loss: 0.2405 | Val AUC-PR: 0.0827 (*)
   Ep 6 | Loss: 0.1909 | Val Loss: 0.2164 | Val AUC-PR: 0.0885 (*)
   Ep 7 | Loss: 0.1804 | Val Loss: 0.2224 | Val AUC-PR: 0.0925 (*)
   Ep 8 | Loss: 0.1827 | Val Loss: 0.2062 | Val AUC-PR: 0.1313 (*)
   Ep 9 | Loss: 0.1732 | Val Loss: 0.2067 | Val AUC-PR: 0.1413 (*)
   Ep 10 | Loss: 0.1747 | Val Loss: 0.2156 | Val AUC-PR: 0.1340 
   Ep 12 | Loss: 0.1719 | Val Loss: 0.2122 | Val AUC-PR: 0.1630 (*)
   Ep 14 | Loss: 0.1593 | Val Loss: 0.2090 | Val AUC-PR: 0.1757 (*)
   Ep 17 | Loss: 0.1508 | Val Loss: 0.2102 | Val AUC-PR: 0.1858 (*)
   Ep 20 | Loss: 0.1480 | Val Loss: 0.2142 | Val AUC-PR: 0.1689 
   Ep 21 | Loss: 0.1410 | Val Loss: 0.1862 | Val AUC-PR: 0.2149 (*)
   Ep 22 | Loss: 0.1377 | V